# E4 — Chuẩn hóa: BN vs LN vs GN × batch size 8 / 32 / 128 (yêu cầu 5.1.4)

**Hướng dẫn người chạy:** bật GPU **T4** (không chọn P100) và Internet → sửa `OWNER` → *Save & Run All* (chạy nền) → tải `outputs/` (file `.zip` in ở cuối) gửi cho điều phối. Không sửa gì khác.

Lưới 3 × 3 × 2 seed = **18 lượt**. SGD momentum 0.9, wd 5e-4, cosine theo bước, không warm-up, 20 epoch. Batch 8 là phần nặng nhất (112 500 bước/lượt). **Nếu điều phối dặn chia theo seed**, chỉ đặt `SEEDS = [0]` hoặc `[1]`.

## Lý do các lựa chọn chính

**BN, LN, GN khác nhau ở tập số dùng để tính mean μ và phương sai σ² (cho mỗi kênh/nhóm):**

| norm | cài đặt | tập số tính (μ, σ) | phụ thuộc batch? |
|---|---|---|---|
| BN | `BatchNorm2d(C)` | từng kênh, gộp trên **cả batch** và H×W | **có** |
| LN | `GroupNorm(1, C)` | **một ảnh**, gộp mọi kênh và H×W | không |
| GN | `GroupNorm(8, C)` | **một ảnh**, từng nhóm C/8 kênh, gộp H×W | không |

Ví dụ ở stage 4 (4×4 = 16 vị trí): BN ở B=8 ước lượng μ, σ của mỗi kênh từ 8×16 = **128** giá trị; ở B=128 là 128×16 = **2 048** giá trị. Ít số hơn ⇒ ước lượng nhiễu hơn ⇒ đây chính là lý do BN suy giảm khi batch nhỏ.

- **Vì sao LN = `GroupNorm(1, C)` mà không dùng `nn.LayerNorm`:** cùng phép chuẩn hóa (thống kê trên (C, H, W) của từng ảnh), nhưng `nn.LayerNorm([C,H,W])` có γ/β theo *từng vị trí không gian* (thêm nhiều tham số, ước tính ~0,6M) và phải biết trước H, W từng lớp. `GroupNorm(1, C)` giữ γ/β theo kênh như BN và GN nên ba loại chỉ khác nhau ở tập số dùng để tính μ, σ — so sánh công bằng.
- **Vì sao GN 8 nhóm:** width 32 làm stage đầu chỉ có 32 kênh; 32 nhóm sẽ thành InstanceNorm (mỗi nhóm 1 kênh).
- **Vì sao linear LR scaling:** `lr = 0.1 × B/128` (0.1 / 0.025 / 0.00625). Nếu giữ lr cố định, batch nhỏ sẽ có nhiễu gradient lớn hơn và việc đổi B lẫn với việc đổi "LR hiệu dụng". `drop_last` cho cả 3 mức batch. Số bước/lượt: 7 020 / 28 120 / 112 500.
- **Vì sao σ_pooled của E4 tính riêng theo batch size:** nhiễu giữa các seed ở B=8 có thể lớn hơn ở B=128 (BN nhiễu hơn), nên ngưỡng "trong nhiễu" phải lấy từ chính batch size đó; gộp chung sẽ làm ngưỡng của B=128 quá rộng hoặc ngưỡng của B=8 quá hẹp.
- **Chẩn đoán D1 (không huấn luyện thêm):** với mô hình BN-b128 vừa huấn luyện, đo độ lệch của mean/var theo kênh tính từ B ảnh (B = 2…128, 200 batch mỗi mức) so với thống kê trên 5 000 ảnh, tại stage 1 và stage 4. Kỳ vọng nhiễu ∝ 1/√B (độ dốc log–log ≈ −½).
- **Ý 2 của lời giải thích (lệch giữa thống kê batch lúc train và running average lúc test)** chỉ nêu bằng lập luận, không đo.

In [ ]:
# ===== CHỈ SỬA CÁC DÒNG TRONG KHỐI NÀY =====
OWNER = "ten_ban"          # tên của bạn (không dấu, không khoảng trắng)
SEEDS = [0, 1]    # nếu điều phối dặn chia theo seed: [0] hoặc [1]
REPO_URL = "https://github.com/<ORG>/<REPO>.git"   # do điều phối cung cấp
COMMIT = "<hash>"                                  # do điều phối cung cấp
# ============================================

import os, sys
if not os.path.exists("repo"):
    !git clone $REPO_URL repo
!cd repo && git checkout $COMMIT
sys.path.insert(0, "repo")
import pandas as pd
import btl
btl.init(owner=OWNER)

In [ ]:
RUNS = {f"E4-{norm}-b{B}": dict(norm=norm, batch_size=B, lr=0.1 * B / 128)
        for norm in ["bn", "ln", "gn"] for B in [8, 32, 128]}

d1_model, d1_seed = None, None
for seed in SEEDS:
    for name, overrides in RUNS.items():
        row, model = btl.run(name, seed, **overrides)
        if name == "E4-bn-b128" and model is not None:
            d1_model, d1_seed = model, seed      # giữ mô hình BN-b128 cho chẩn đoán D1

In [ ]:
# Chẩn đoán D1: nhiễu thống kê batch của BN theo B (không huấn luyện thêm)
if d1_model is not None:
    d1 = btl.bn_noise(d1_model).assign(seed=d1_seed)
    d1.to_csv(f"{btl.OUT_DIR}/d1.csv", index=False)
    print(d1.to_string(index=False))
else:
    print("Không có mô hình BN-b128 completed trên máy này -> bỏ qua D1 (chạy ở máy có seed chứa E4-bn-b128)")

In [ ]:
runs = pd.read_csv(f"{btl.OUT_DIR}/runs.csv")
print(runs[["config", "seed", "status", "final_test_acc", "duration_min"]].to_string(index=False))
bad = runs[runs.status != "completed"]
print(f"\nTổng {len(runs)} lượt, {len(bad)} lượt diverged/crashed" + (" -> ĐỌC KỸ cột notes trong runs.csv" if len(bad) else ""))
btl.zip_outputs("04_normalization")   # gửi file .zip này cho điều phối